# 安装与运行示例

此 notebook 提供一个可复制的快速上手示例，演示：

1. 配置 `CAi/.env`
2. 安装项目依赖
3. 启动工具后端（FastAPI）并自检 `/health`
4. 用 `A1pro` 直接发一条 prompt

Notebook 中带 `!` 的行会在 shell 下执行；也可以在终端中逐条运行。

## 前提与假设

- 假设你在项目根目录 `CAi_copilot/` 下运行此 notebook。
- 准备好 LLM API key（Anthropic、OpenAI、DeepSeek 任选其一；也可指向任意 OpenAI 兼容端点）。下面的 cell 会写入 `.env` 模板到 `CAi/.env`，首次运行前请填好真实的 key。
- 工具后端每个子工具运行在独立 conda 环境里，**首次安装较慢**，建议在终端里跑 `install_all.sh`。

In [ ]:
# 1. 写入 .env 模板（已存在则跳过，避免覆盖你已有的 key）
from pathlib import Path

env_path = Path('CAi') / '.env'
env_path.parent.mkdir(parents=True, exist_ok=True)

env_template = '''# -------------------------------------------------------------
# CAi env file — LLM_SOURCE 可留空，程序会根据 LLM_MODEL 自动识别
#   claude-*    → Anthropic   (env: ANTHROPIC_API_KEY 或 LLM_API_KEY)
#   gpt-*/o1-*  → OpenAI      (env: OPENAI_API_KEY 或 LLM_API_KEY)
#   deepseek-*  → DeepSeek    (env: DEEPSEEK_API_KEY 或 LLM_API_KEY)
#   other + LLM_BASE_URL → Custom (任意 OpenAI 兼容端点)
# -------------------------------------------------------------
LLM_MODEL=claude-sonnet-4-5-20250929
LLM_API_KEY=your_api_key_here
# LLM_SOURCE=              # 一般留空；需要强制时填 OpenAI / Anthropic / DeepSeek / Custom
# LLM_BASE_URL=            # 仅 Custom 需要（例如 http://localhost:8000/v1）
# LLM_TEMPERATURE=0.7

TOOL_SERVER_HOST=0.0.0.0
TOOL_SERVER_PORT=8001
'''

if env_path.exists():
    print(f'{env_path} 已存在，跳过写入。如需重置，请手动删除后重跑此 cell。')
else:
    env_path.write_text(env_template, encoding='utf-8')
    print(f'模板 .env 已写入: {env_path}\n请填入真实 LLM_API_KEY 后再继续。')

In [ ]:
# 2. 安装基础依赖
import sys

!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install -e .

print('\nCAi 包已安装。可选：pip install -e ".[dev]" 可以装上 pytest / pytest-asyncio。')

In [ ]:
# 3. 工具后端：每个子工具跑在独立 conda 环境中，按需安装
#   - 完整安装：bash install_all.sh
#   - 按名安装：bash install_all.sh vina scscore toxicity
# 这一步建议在终端里跑，notebook 里只是展示命令。
from pathlib import Path

install_sh = Path('CAi') / 'toolkit' / 'server' / 'install_all.sh'
if install_sh.exists():
    print(f'Found: {install_sh}')
    print('\n推荐在终端中运行：')
    print(f'    bash {install_sh}')
    print('\n或只安装需要的工具：')
    print(f'    bash {install_sh} vina scscore toxicity')
else:
    print(f'install_all.sh not found at {install_sh}')

In [ ]:
# 4. 启动工具后端（FastAPI）
# 推荐在终端里作为前台进程运行，方便查看日志：
#     python -m CAi.toolkit.server.app
# 启动时会打印 banner，列出已加载的工具、工作目录和监听端口。
#
# 如果你一定要在 notebook 里后台启动（内核重启后会被杀）：
print('推荐（在独立终端中运行，便于看 banner 与日志）:')
print('    python -m CAi.toolkit.server.app')
print('\n或后台（nohup，Linux/macOS 可用）：')
print('    !nohup python -m CAi.toolkit.server.app > tool_server.log 2>&1 &')

In [ ]:
# 5. 自检：调 /health 确认 server 起来 + 工具已加载
import requests

base = 'http://127.0.0.1:8001'
try:
    r = requests.get(f'{base}/health', timeout=5)
    r.raise_for_status()
    data = r.json()
    print('Status:', data.get('status'))
    print('Workspace:', data.get('workspace'))
    print(f'Loaded {len(data.get("tools", []))} tools:')
    for t in data.get('tools', []):
        print(f'  - {t}')
except Exception as e:
    print(f'Unable to reach tool server at {base}: {e}')
    print('\n请先启动后端：python -m CAi.toolkit.server.app')

## 用 `A1pro` 发送一条 prompt

`A1pro` 会根据 `LLM_MODEL` 自动识别 provider（claude-* / gpt-* / deepseek-* 直接走官方端点，其它走 Custom + `LLM_BASE_URL`）。

In [ ]:
from CAi.CAi_agent import A1pro
from CAi.config import (
    LLM_API_KEY,
    LLM_BASE_URL,
    LLM_MODEL,
    LLM_SOURCE,
    LLM_TEMPERATURE,
)

# 常见 LLM_MODEL 的写法（在 .env 里配置即可，这里仅供参考）:
#   LLM_MODEL=claude-sonnet-4-5-20250929
#   LLM_MODEL=gpt-4o-mini
#   LLM_MODEL=deepseek-chat
#   LLM_MODEL=qwen-plus               (需同时配 LLM_BASE_URL 指向 OpenAI 兼容端点)

agent = A1pro(
    llm=LLM_MODEL,
    source=LLM_SOURCE,        # None → 根据 LLM_MODEL 前缀自动识别
    base_url=LLM_BASE_URL,    # 只有 Custom 需要
    api_key=LLM_API_KEY,
    temperature=LLM_TEMPERATURE,
    auto_load_tools=True,     # 自动扫描 CAi.toolkit 中的工具
    auto_load_skills=False,   # 需要 skills 时打开
)

prompt = '请介绍一下你自己，并列举你可以使用的工具。'

# agent.run() 返回 (log, final_content)
log, final = agent.run(prompt)
print(final)

## 启动 Web UI

在项目根目录（`CAi_copilot/`）下运行：

```bash
python CAi/main.py
```

这会启动 FastAPI 后端（默认 `http://localhost:7001`）并托管静态前端 `CAi/web_ui/frontend/`。浏览器打开该地址即可交互。

要更换模型或修改默认参数，编辑 `CAi/.env`（首选）或直接改 `CAi/config.py` 中的默认值。

## 故障排查

快速自检：

```bash
curl -s http://127.0.0.1:8001/health | jq       # 工具后端
curl -s http://127.0.0.1:7001/api/health | jq    # Web 后端（需 main.py 已启动）
```

常见问题：
- **端口被占用**：`lsof -i :8001` 或 `ss -ltnp` 查进程，必要时换端口（修改 `.env` 的 `TOOL_SERVER_PORT`）。
- **工具列表为空**：检查 `CAi/toolkit/server/tools/<tool>/config.json` 是否存在，以及对应 conda env 是否安装好。
- **LLM 请求失败**：先确认 `LLM_MODEL` 与 `LLM_API_KEY` 匹配，并且（对 Custom）`LLM_BASE_URL` 可达。运行 `python -c "from CAi.toolkit.client import ping; print(ping())"` 做一次最小自检。